# Formulacion matematica del TSI provisional
Esta subfase formaliza el indice calculado en `03_analisis_datos`. El TSI se normaliza dentro de cada fuente y metrica; no es una escala internacional unica mientras las unidades y semanticas sean diferentes.

In [2]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
ANALYSIS = PROJECT_ROOT / 'data' / '03_analysis_output'
OUTPUT_DIR = PROJECT_ROOT / 'data' / '04_model_output' / 'tsi'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
method = json.loads((ANALYSIS / 'indicador_tsi' / 'metodo.json').read_text(encoding='utf-8'))
profiles = pd.read_csv(ANALYSIS / 'comparacion_fuentes' / 'perfil_comparativo_fuentes.csv')
daily_tsi = pd.read_csv(ANALYSIS / 'indicador_tsi' / 'tsi_provisional_diario.csv')
print(method)
display(profiles[['fuente', 'metrica_principal', 'familia_metrica', 'media_valor_principal', 'minimo_valor_principal', 'maximo_valor_principal']])

{'tipo': 'provisional', 'normalizacion': 'min_max_dentro_de_cada_fuente_y_metrica', 'escala': '0_100', 'fuentes_mezcladas': False, 'filas_evaluadas': 41770880, 'limitacion': 'requiere validacion de unidades, sensibilidad a extremos y calibracion local'}


,fuente,metrica_principal,familia_metrica,media_valor_principal,minimo_valor_principal,maximo_valor_principal
0,istanbul_traffic_index,indice_congestion,indice_congestion,27.830846,1.083916,59.428571
1,metr_la,velocidad_sensor,velocidad,53.719021,0.000000,70.000000
2,madrid_traffic_points,intensidad,intensidad,366.029942,0.000000,26327.000000
3,pems_bay,velocidad_sensor,velocidad,62.619574,0.000000,85.100000
4,sao_paulo_cet,congestion_longitud_m,congestion_longitud,1453.528826,51.000000,23200.000000


## Formula
Para cada fuente `s` y metrica `m`, se calculan limites `min(s,m)` y `max(s,m)` usando el conjunto de referencia de esa misma fuente.

Si un valor alto representa mejor estabilidad, se usa:

`TSI(s,m,t) = 100 * (x(s,m,t) - min(s,m)) / (max(s,m) - min(s,m))`

Si un valor bajo representa mejor estabilidad, se usa:

`TSI(s,m,t) = 100 * (max(s,m) - x(s,m,t)) / (max(s,m) - min(s,m))`

El resultado se limita al intervalo `[0, 100]`. Un valor cercano a 100 representa una mejor posicion relativa dentro de la fuente; no significa que una ciudad sea objetivamente mas estable que otra.

In [3]:
DIRECTIONS = {
    'indice_congestion': 'lower_is_better',
    'velocidad': 'higher_is_better',
    'intensidad': 'lower_is_better',
    'congestion_longitud': 'lower_is_better',
}
profiles['direccion_tsi'] = profiles['familia_metrica'].map(DIRECTIONS)
profiles['rango_referencia'] = profiles['maximo_valor_principal'] - profiles['minimo_valor_principal']
profiles['normalizacion_valida'] = profiles['rango_referencia'] > 0
display(profiles[['fuente', 'metrica_principal', 'direccion_tsi', 'minimo_valor_principal', 'maximo_valor_principal', 'rango_referencia', 'normalizacion_valida']])
display(daily_tsi.groupby('fuente', as_index=False).agg(dias=('fecha', 'nunique'), tsi_medio=('tsi_provisional_media', 'mean'), tsi_minimo=('tsi_provisional_minimo', 'min'), tsi_maximo=('tsi_provisional_maximo', 'max'))) 

,fuente,metrica_principal,direccion_tsi,minimo_valor_principal,maximo_valor_principal,rango_referencia,normalizacion_valida
0,istanbul_traffic_index,indice_congestion,lower_is_better,1.083916,59.428571,58.344655,True
1,metr_la,velocidad_sensor,higher_is_better,0.000000,70.000000,70.000000,True
2,madrid_traffic_points,intensidad,lower_is_better,0.000000,26327.000000,26327.000000,True
3,pems_bay,velocidad_sensor,higher_is_better,0.000000,85.100000,85.100000,True
4,sao_paulo_cet,congestion_longitud_m,lower_is_better,51.000000,23200.000000,23149.000000,True


,fuente,dias,tsi_medio,tsi_minimo,tsi_maximo
0,istanbul_traffic_index,3332,54.157017,4.896922e-11,100.0
1,madrid_traffic_points,31,98.609834,0.000000e+00,100.0
2,metr_la,119,76.741459,0.000000e+00,100.0
3,pems_bay,181,73.584399,0.000000e+00,100.0
4,sao_paulo_cet,6094,94.243199,0.000000e+00,100.0


In [4]:
formula_report = {
    'nombre': 'Traffic Stability Index provisional',
    'escala': [0, 100],
    'unidad_de_comparacion': 'relativa_dentro_de_fuente_y_metrica',
    'normalizacion': 'min_max',
    'direcciones': DIRECTIONS,
    'fuentes_mezcladas': False,
    'regla_valor_alto': 'mejor estabilidad relativa dentro de la fuente',
    'uso_actual': 'objetivo de pronostico exploratorio del siguiente dia',
    'no_debe_usarse_como': ['verdad universal de estabilidad', 'etiqueta de precolapso', 'comparacion directa entre unidades distintas'],
    'filas_evaluadas': int(method['filas_evaluadas']),
}
(OUTPUT_DIR / 'formulacion_tsi_provisional.json').write_text(json.dumps(formula_report, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(formula_report, indent=2, ensure_ascii=False))

{
  "nombre": "Traffic Stability Index provisional",
  "escala": [
    0,
    100
  ],
  "unidad_de_comparacion": "relativa_dentro_de_fuente_y_metrica",
  "normalizacion": "min_max",
  "direcciones": {
    "indice_congestion": "lower_is_better",
    "velocidad": "higher_is_better",
    "intensidad": "lower_is_better",
    "congestion_longitud": "lower_is_better"
  },
  "fuentes_mezcladas": false,
  "regla_valor_alto": "mejor estabilidad relativa dentro de la fuente",
  "uso_actual": "objetivo de pronostico exploratorio del siguiente dia",
  "no_debe_usarse_como": [
    "verdad universal de estabilidad",
    "etiqueta de precolapso",
    "comparacion directa entre unidades distintas"
  ],
  "filas_evaluadas": 41770880
}


## Decision metodologica
El TSI esta formalizado y reproducido como indicador relativo por fuente. El modelo Random Forest pronostica este indicador provisional; el proxy de deterioro anticipado y la deteccion de anomalías son experimentos separados y no deben presentarse como parte de la formula TSI.